# BuildersVault Social Services Hackathon Track 1 Quickstart

**Track 1: Inter-Org Referral and Care Coordination**

**Standards in scope:** HIFIS 4 referral module, BC Coordinated Access, AIRS taxonomy, VI-SPDAT acuity, PIPA / FOIPPA / OCAP consent law.

## What this notebook does in 30 seconds

Loads the 7 synthetic datasets, runs basic schema and quality checks, builds a golden joined view that most teams will build on top of, does some quick EDA, and runs a rule-based duplicate-detection baseline so you have a number to beat.

## What the data represents

- **9 mock Victoria-area social service orgs** modeled on real HIFIS 4 deployments plus BC Coordinated Access partners.
- **800 clients** with realistic demographic messiness (missing DOBs, nickname variants, partial addresses, OCAP-flagged Indigenous clients).
- **3,000 referrals** across the full HIFIS lifecycle (submitted, accepted, declined, completed, withdrawn, no-show).
- **10,000 service encounters** spanning shelter nights, case-management sessions, outreach contacts, health visits.
- **5,000 consent records** with active, expired, withdrawn, and scope-mismatched states.
- **Ground-truth duplicate flags** with 300 true positives and 200 decoy near-duplicates so you can actually score your entity-resolution work.

## 4 concrete challenges to pick from

1. **Duplicate detection and merge** across agencies without violating OCAP (a client can exist in 3 orgs under 3 spellings).
2. **Consent-gap surfacing** so caseworkers see when data sharing is unlawful before they act.
3. **Referral lifecycle bottlenecks** so coordinators see where referrals stall or die.
4. **Falling-through-the-gaps risk detection** for chronically-vulnerable clients who are losing touch with the system.

Pick one. Go deep. Judges care about domain grounding more than model complexity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from difflib import SequenceMatcher

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data/raw')

expected_files = [
    'orgs.parquet',
    'clients.parquet',
    'referrals.parquet',
    'encounters.parquet',
    'consent.parquet',
    'dsa.parquet',
    'duplicate_flags.parquet',
]

missing = [f for f in expected_files if not (DATA_DIR / f).exists()]
if missing:
    print('Missing data files:', missing)
    print('Run: python ../generator/generate.py')
else:
    print(f'All {len(expected_files)} data files present in {DATA_DIR.resolve()}')


In [ ]:
orgs = pd.read_parquet(DATA_DIR / 'orgs.parquet')
clients = pd.read_parquet(DATA_DIR / 'clients.parquet')
referrals = pd.read_parquet(DATA_DIR / 'referrals.parquet')
encounters = pd.read_parquet(DATA_DIR / 'encounters.parquet')
consent = pd.read_parquet(DATA_DIR / 'consent.parquet')
dsa = pd.read_parquet(DATA_DIR / 'dsa.parquet')
dup_flags = pd.read_parquet(DATA_DIR / 'duplicate_flags.parquet')

tables = {
    'orgs': orgs,
    'clients': clients,
    'referrals': referrals,
    'encounters': encounters,
    'consent': consent,
    'dsa': dsa,
    'duplicate_flags': dup_flags,
}

for name, df in tables.items():
    print(f'{name:20s} shape = {df.shape}')


In [ ]:
rows = []
for name, df in tables.items():
    for col in df.columns:
        nulls = df[col].isna().sum()
        rows.append({
            'table': name,
            'column': col,
            'dtype': str(df[col].dtype),
            'null_count': int(nulls),
            'null_pct': round(100.0 * nulls / max(len(df), 1), 2),
        })

null_summary = pd.DataFrame(rows)
print('Top 15 columns by null percentage:')
null_summary.sort_values('null_pct', ascending=False).head(15)


In [ ]:
expected_schema = {
    'orgs': ['org_id', 'org_name', 'org_type', 'airs_taxonomy_codes', 'city'],
    'clients': ['client_id', 'first_name', 'last_name', 'dob', 'address_line1',
                'indigenous_flag', 'ocap_protected', 'current_consent_id', 'vi_spdat_score'],
    'referrals': ['referral_id', 'client_id', 'referring_org_id', 'receiving_org_id',
                  'status', 'submitted_at', 'decision_at', 'completed_at', 'reason_code'],
    'encounters': ['encounter_id', 'client_id', 'org_id', 'encounter_type',
                   'occurred_at', 'notes'],
    'consent': ['consent_id', 'client_id', 'granting_org_id', 'status',
                'sharing_scope_type', 'purpose_codes', 'effective_at', 'expires_at'],
    'dsa': ['dsa_id', 'org_a_id', 'org_b_id', 'signed_at', 'scope'],
    'duplicate_flags': ['pair_id', 'client_id_a', 'client_id_b', 'is_true_duplicate'],
}

all_pass = True
for name, cols in expected_schema.items():
    df = tables[name]
    missing_cols = [c for c in cols if c not in df.columns]
    status = 'OK' if not missing_cols else 'FAIL'
    if missing_cols:
        all_pass = False
    print(f'[{status}] {name:20s} expected {len(cols):2d} cols, missing: {missing_cols or "none"}')

assert all_pass, 'Schema check failed. Regenerate the dataset or update expected_schema above.'
print('\nAll expected columns present.')


In [ ]:
referring = orgs[['org_id', 'org_name', 'org_type']].rename(columns={
    'org_id': 'referring_org_id',
    'org_name': 'referring_org_name',
    'org_type': 'referring_org_type',
})

receiving = orgs[['org_id', 'org_name', 'org_type']].rename(columns={
    'org_id': 'receiving_org_id',
    'org_name': 'receiving_org_name',
    'org_type': 'receiving_org_type',
})

client_slim = clients[['client_id', 'first_name', 'last_name', 'dob',
                       'indigenous_flag', 'ocap_protected', 'vi_spdat_score',
                       'current_consent_id']]

consent_slim = consent[['consent_id', 'status', 'sharing_scope_type',
                        'effective_at', 'expires_at']].rename(columns={
    'consent_id': 'current_consent_id',
    'status': 'consent_status',
    'sharing_scope_type': 'consent_scope',
    'effective_at': 'consent_effective_at',
    'expires_at': 'consent_expires_at',
})

referrals_enriched = (
    referrals
    .merge(client_slim, on='client_id', how='left')
    .merge(referring, on='referring_org_id', how='left')
    .merge(receiving, on='receiving_org_id', how='left')
    .merge(consent_slim, on='current_consent_id', how='left')
)

curated_cols = [
    'referral_id', 'client_id', 'first_name', 'last_name',
    'referring_org_name', 'referring_org_type',
    'receiving_org_name', 'receiving_org_type',
    'status', 'submitted_at', 'decision_at', 'completed_at',
    'vi_spdat_score', 'indigenous_flag', 'ocap_protected',
    'consent_status', 'consent_scope', 'consent_expires_at',
]

print(f'referrals_enriched shape: {referrals_enriched.shape}')
referrals_enriched[curated_cols].head(10)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Referral status breakdown
status_counts = referrals['status'].value_counts()
axes[0, 0].bar(status_counts.index, status_counts.values, color='steelblue')
axes[0, 0].set_title('Referral status breakdown')
axes[0, 0].set_ylabel('count')
axes[0, 0].tick_params(axis='x', rotation=30)

# 2. Referral volume by referring org type
by_ref_type = referrals_enriched['referring_org_type'].value_counts()
axes[0, 1].bar(by_ref_type.index, by_ref_type.values, color='darkorange')
axes[0, 1].set_title('Referral volume by referring org type')
axes[0, 1].set_ylabel('count')
axes[0, 1].tick_params(axis='x', rotation=30)

# 3. Consent status mix
consent_mix = consent['status'].value_counts()
axes[0, 2].pie(consent_mix.values, labels=consent_mix.index, autopct='%1.1f%%',
               startangle=90)
axes[0, 2].set_title('Consent status mix')

# 4. VI-SPDAT score distribution
vi_scores = clients['vi_spdat_score'].dropna()
axes[1, 0].hist(vi_scores, bins=range(0, int(vi_scores.max()) + 2),
                color='seagreen', edgecolor='white')
axes[1, 0].set_title('Client acuity (VI-SPDAT) distribution')
axes[1, 0].set_xlabel('score')
axes[1, 0].set_ylabel('clients')

# 5. Referral lifecycle time (submitted to completed or declined)
closed = referrals_enriched[referrals_enriched['status'].isin(
    ['completed', 'declined'])].copy()
closed['submitted_at'] = pd.to_datetime(closed['submitted_at'], errors='coerce')
closed['decision_at'] = pd.to_datetime(closed['decision_at'], errors='coerce')
closed['lifecycle_days'] = (closed['decision_at'] - closed['submitted_at']
                             ).dt.total_seconds() / 86400.0
lifecycle = closed['lifecycle_days'].dropna()
lifecycle = lifecycle[lifecycle > 0]
axes[1, 1].hist(lifecycle, bins=40, color='purple', edgecolor='white')
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Referral lifecycle (days, log scale)')
axes[1, 1].set_xlabel('days submitted to decision')

# 6. Empty slot used for a quick summary card
axes[1, 2].axis('off')
summary_text = (
    f"Totals\n"
    f"orgs: {len(orgs):>6,}\n"
    f"clients: {len(clients):>6,}\n"
    f"referrals: {len(referrals):>6,}\n"
    f"encounters: {len(encounters):>6,}\n"
    f"consent: {len(consent):>6,}\n"
    f"dsa: {len(dsa):>6,}\n"
    f"dup_flags: {len(dup_flags):>6,}"
)
axes[1, 2].text(0.05, 0.5, summary_text, fontfamily='monospace', fontsize=12,
                verticalalignment='center')

plt.tight_layout()
plt.show()


In [ ]:
# Baseline: rule-based duplicate detector using Soundex-style bucketing + string similarity

def soundex(name):
    if not isinstance(name, str) or not name:
        return '0000'
    name = name.upper()
    first = name[0]
    mapping = {'B': '1', 'F': '1', 'P': '1', 'V': '1',
               'C': '2', 'G': '2', 'J': '2', 'K': '2', 'Q': '2', 'S': '2', 'X': '2', 'Z': '2',
               'D': '3', 'T': '3',
               'L': '4',
               'M': '5', 'N': '5',
               'R': '6'}
    tail = ''.join(mapping.get(c, '') for c in name[1:])
    # collapse doubles
    collapsed = ''
    prev = ''
    for c in tail:
        if c != prev:
            collapsed += c
        prev = c
    return (first + collapsed + '000')[:4]


def sim(a, b):
    if not isinstance(a, str) or not isinstance(b, str):
        return 0.0
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


work = clients[['client_id', 'first_name', 'last_name', 'dob', 'address_line1']].copy()
work['last_soundex'] = work['last_name'].apply(soundex)

pairs = []
for _, bucket in work.groupby('last_soundex'):
    if len(bucket) < 2 or len(bucket) > 80:
        continue  # skip singletons and pathologically large buckets
    recs = bucket.to_dict('records')
    for i in range(len(recs)):
        for j in range(i + 1, len(recs)):
            a, b = recs[i], recs[j]
            first_sim = sim(a['first_name'], b['first_name'])
            dob_match = 1.0 if pd.notna(a['dob']) and pd.notna(b['dob']) and a['dob'] == b['dob'] else 0.0
            addr_sim = sim(a.get('address_line1'), b.get('address_line1'))
            score = 0.5 * first_sim + 0.3 * dob_match + 0.2 * addr_sim
            if score >= 0.75:
                pairs.append({
                    'client_id_a': a['client_id'],
                    'client_id_b': b['client_id'],
                    'score': round(score, 3),
                })

baseline = pd.DataFrame(pairs)
print(f'Baseline predicted {len(baseline)} duplicate pairs')

# Normalize pair direction so (a,b) == (b,a) for join
def keyify(row, a_col, b_col):
    a, b = row[a_col], row[b_col]
    return tuple(sorted([a, b]))

gt = dup_flags.copy()
gt['pair_key'] = gt.apply(lambda r: keyify(r, 'client_id_a', 'client_id_b'), axis=1)
baseline['pair_key'] = baseline.apply(lambda r: keyify(r, 'client_id_a', 'client_id_b'), axis=1)

gt_true_keys = set(gt[gt['is_true_duplicate']]['pair_key'])
gt_all_keys = set(gt['pair_key'])
pred_keys = set(baseline['pair_key'])

tp = len(pred_keys & gt_true_keys)
fp = len(pred_keys - gt_true_keys)
fn = len(gt_true_keys - pred_keys)
precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-9)

print(f'\nBaseline vs ground truth:')
print(f'  true positives:  {tp}')
print(f'  false positives: {fp}')
print(f'  false negatives: {fn}')
print(f'  precision: {precision:.3f}')
print(f'  recall:    {recall:.3f}')
print(f'  f1:        {f1:.3f}')
print('\nBeat this.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Confusion matrix heatmap for baseline vs ground truth
tn_approx = len(gt_all_keys) - tp - fp - fn  # only defined within the gt pair universe
cm = np.array([[tp, fn],
               [fp, max(tn_approx, 0)]])
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['predicted dup', 'predicted non-dup'])
axes[0].set_yticklabels(['actual dup', 'actual non-dup'])
axes[0].set_title('Baseline duplicate detector (within flagged pairs)')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black',
                     fontsize=14)
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# (b) Top 10 orgs by consent gap rate
enc = encounters.merge(
    clients[['client_id', 'current_consent_id']], on='client_id', how='left'
).merge(
    consent[['consent_id', 'status', 'expires_at']].rename(
        columns={'consent_id': 'current_consent_id'}),
    on='current_consent_id', how='left'
)
enc['occurred_at'] = pd.to_datetime(enc['occurred_at'], errors='coerce')
enc['expires_at'] = pd.to_datetime(enc['expires_at'], errors='coerce')
enc['has_gap'] = (
    enc['status'].isna()
    | (enc['status'].isin(['expired', 'withdrawn']))
    | ((enc['expires_at'].notna()) & (enc['occurred_at'] > enc['expires_at']))
)

gap_by_org = enc.merge(orgs[['org_id', 'org_name']], on='org_id', how='left')
gap_stats = gap_by_org.groupby('org_name').agg(
    encounters=('encounter_id', 'count'),
    gaps=('has_gap', 'sum'),
)
gap_stats['gap_rate'] = gap_stats['gaps'] / gap_stats['encounters']
gap_stats = gap_stats.sort_values('gap_rate', ascending=False).head(10)

axes[1].barh(gap_stats.index[::-1], gap_stats['gap_rate'].values[::-1], color='crimson')
axes[1].set_xlabel('consent gap rate (encounters without active matching consent)')
axes[1].set_title('Top 10 orgs by consent gap rate')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()


## Extension ideas and rubric alignment

The baseline above is deliberately simple. Here is where to go next, mapped to the 4 judging themes.

### Build ideas

- **ML-based entity resolution** with Splink or recordlinkage. Learn agreement weights from the seeded duplicate pairs. Handle Indigenous name variants and nicknames without leaking OCAP-protected attributes into public features.
- **Embedding-based name matching** using sentence-transformers on `first_name + last_name + dob + address`. Catches phonetic and transliteration variants that Soundex misses.
- **Live consent-gap dashboard** (Streamlit or Next.js) that shows caseworkers a red banner the moment they pull a record whose consent is expired, withdrawn, or scope-mismatched.
- **Referral decline predictor** that estimates probability of `status = declined` at submission time from client acuity, referring/receiving org pair, and encounter history.
- **Care-coordination API** that exposes a single endpoint `/clients/{id}/safe-view` which automatically redacts fields based on OCAP and `sharing_scope_type`.
- **Falling-through-the-gaps risk score** combining VI-SPDAT, days since last encounter, consent expiry, and referral decline history.

### Rubric alignment

| Judging theme | What moves the needle |
| --- | --- |
| **Problem Fit** | You can name the operational pain (duplicate-client triage, unlawful data sharing, silent referral death) and show your solution removes it. Include a before/after caseworker workflow. |
| **Technical Merit** | Beat the baseline F1 on duplicates. Show ROC curves, ablations, or calibration plots. No magic numbers, no hand-tuned thresholds that happen to fit the seeded pairs. |
| **Social Services Domain Grounding** | References to HIFIS 4 field names, AIRS taxonomy codes, VI-SPDAT bands, and correct handling of OCAP and PIPA/FOIPPA. Judges include people who have operated these systems. |
| **Production Readiness** | Tests, schema validation, a runnable demo, and a clear story on privacy, observability, and failure modes. A notebook alone does not qualify. |

### Before you submit

- Re-read `../docs/problem-framing.md` once more. Make sure your solution honors every privacy constraint listed there.
- Record a 3 minute walkthrough. Judges watch everything at 1.5x. Be clear and specific.
- Put your repo URL, demo URL, and a 150-word abstract in your submission form.